# 01. Clean Reddit submissions

Takes the raw r/WomensHealth submissions dump, keeps English self-posts, cleans the text, and tokenizes it. The output of this notebook feeds every other notebook in this folder, so run it first.

**Run from `notebooks/Reddit_Data/`.** All paths are relative to that folder.

**Input:** `data/interim/subreddits/WomensHealth/submissions`
**Output:** `data/interim/cleaned_submissions.parquet`

The raw archive ships as `WomensHealth_submissions.zst` in `data/raw/subreddits25/`. Unpack it before running this notebook. The unpacked file is line-delimited JSON with one post per line and no file extension.

Dependencies are in the project `requirements.txt`.

## Load raw submissions

In [ ]:
import pandas as pd

RAW_PATH = "../../data/interim/subreddits/WomensHealth/submissions"

# The dump has ~100 columns, most of which are Reddit API metadata we never use.
# Keeping only what we need here saves memory and avoids a large drop list later.
KEEP_COLS = [
    "created_utc", "created", "title", "selftext", "author", "is_self",
    "num_comments", "score", "ups", "upvote_ratio", "over_18",
    "total_awards_received",
]

# Read in chunks so the 370MB file does not have to sit in memory all at once.
chunks = []
for chunk in pd.read_json(RAW_PATH, lines=True, chunksize=50_000):
    chunk = chunk[chunk["is_self"] == True]
    chunks.append(chunk[[c for c in KEEP_COLS if c in chunk.columns]])

df = pd.concat(chunks, ignore_index=True)
df.info()

## Keep English posts

`langdetect` is nondeterministic by default, so we set the seed to make this step reproducible. This is the slowest cell in the notebook.

In [ ]:
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 0

def is_english(text):
    try:
        return detect(str(text)[:100]) == "en"
    except Exception:
        return False

df = df[df["title"].apply(is_english) & df["selftext"].apply(is_english)]
df.info()

In [ ]:
df.to_parquet("../../data/interim/english_submissions.parquet")

## Clean titles

In [ ]:
# Placeholder titles left behind by deleted or moderated posts
df = df[df["title"] != "[deleted by user]"]
df = df[df["title"] != "[ Removed by moderator ]"]

df["title"] = df["title"].str.lower()

# Only strip question and exclamation marks. Other punctuation is left in
# because titles are short and over-cleaning them loses meaning.
df["title"] = (df["title"]
               .str.replace(r"[?!]", " ", regex=True)
               .str.replace(r"\s+", " ", regex=True)
               .str.strip())

In [ ]:
print("Top 15 most common titles:")
print(df["title"].value_counts().head(15))

## Clean selftext

Some users post the same text more than once. We keep the copy with the most comments, since that is the version the community actually engaged with.

In [ ]:
dupe_counts = (df.groupby(["author", "selftext"])
                 .size()
                 .reset_index(name="count")
                 .query("count > 1")
                 .sort_values("count", ascending=False))
dupe_counts.head(10)

In [ ]:
df = (df.sort_values("num_comments", ascending=False)
        .drop_duplicates(subset=["author", "selftext"], keep="first")
        .sort_index())
df.info()

In [ ]:
import re
import string
from bs4 import BeautifulSoup

def clean_text(text):
    text = str(text).lower()
    text = BeautifulSoup(text, "html.parser").get_text()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\W+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["selftext"] = df["selftext"].apply(clean_text)

In [ ]:
# Boilerplate left behind when Reddit removes a post. Punctuation is already
# stripped at this point, which is why it reads as one run-on string.
df = df[df["selftext"] != "removed by reddit on account of violating the content policyhelpcontentpolicy"]

print("Top 15 most common selftext:")
print(df["selftext"].value_counts().head(15))

## Tokenize and remove stopwords

Standard English stopwords miss a lot of forum filler, so we add our own list on top. These are words that show up constantly in health posts but say nothing about the topic.

In [ ]:
import nltk
from nltk.tokenize import word_tokenize

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

df["title_tokens"] = df["title"].apply(word_tokenize)
df["selftext_tokens"] = df["selftext"].apply(word_tokenize)

In [ ]:
from nltk.corpus import stopwords

nltk.download("stopwords", quiet=True)

stop_words = set(stopwords.words("english"))

# Common in forum posts but carries no topic signal
custom_stops = {
    "im", "ive", "id", "dont", "didnt", "doesnt", "cant", "wont", "isnt",
    "like", "get", "got", "getting", "go", "going", "went", "know", "think",
    "feel", "feeling", "felt", "really", "also", "even", "still", "would",
    "could", "should", "much", "many", "lot", "thing", "things", "something",
    "anything", "anyone", "someone", "want", "wanted", "need", "say", "said",
    "see", "one", "two", "first", "time", "day", "days", "week", "weeks",
    "month", "months", "year", "years", "ago", "back", "since", "never",
    "ever", "always", "take", "taking", "took", "make", "made", "well",
    "good", "bad", "new", "way", "help", "thanks", "thank", "please",
    "question", "advice", "else", "experience", "post", "edit",
    "update", "title", "sorry", "long", "story", "started", "start",
    "trying", "tried", "maybe", "sure", "right", "around", "every",
    "last", "next", "used", "use", "using",
}

all_stops = stop_words | custom_stops

def extract_keywords(tokens):
    # Drop stopwords and anything two characters or shorter
    return [t for t in tokens if t not in all_stops and len(t) > 2]

df["title_keywords"] = df["title_tokens"].apply(extract_keywords)
df["selftext_keywords"] = df["selftext_tokens"].apply(extract_keywords)

In [ ]:
df.to_parquet("../../data/interim/cleaned_submissions.parquet")
print(f"{len(df)} posts saved")